# 01 - EDA Rossmann Store Sales

This notebook performs initial exploratory data analysis (EDA), generates core figures, and creates baseline cleaned datasets for downstream modeling.

## Setup Notes
- This section imports packages and configures plotting defaults.
- Keep these settings consistent across notebooks to ensure comparable visuals.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

In [ ]:
RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
FIG_DIR = Path('../reports/figures')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Data Loading and Sanity Check
- We load `train`, `test`, and `store` directly from `data/raw/`.
- Shape checks confirm row/column counts before any transformation.


In [ ]:
# Read base competition files with parsed dates for time-aware analysis
train = pd.read_csv(RAW_DIR / 'train.csv', parse_dates=['Date'])
test = pd.read_csv(RAW_DIR / 'test.csv', parse_dates=['Date'])
store = pd.read_csv(RAW_DIR / 'store.csv')

print('train:', train.shape)
print('test :', test.shape)
print('store:', store.shape)

In [ ]:
train.head()

In [ ]:
train.info()

## Missing Data Audit
- We compute missing-value ratios to identify variables that need imputation.
- Store metadata often has sparse competition/promo timing fields.


In [ ]:
# Profile missingness to prioritize imputation strategy for modeling
missing_train = train.isna().mean().sort_values(ascending=False)
missing_store = store.isna().mean().sort_values(ascending=False)

print('Top train missing columns:')
print(missing_train[missing_train > 0].head(10))
print('\nTop store missing columns:')
print(missing_store[missing_store > 0].head(10))

In [ ]:
train_m = train.merge(store, on='Store', how='left')
test_m = test.merge(store, on='Store', how='left')

print('Merged train:', train_m.shape)
print('Merged test :', test_m.shape)

## Feature Preparation Rationale
- Date decomposition captures seasonality and calendar effects.
- Numeric sparse fields are zero-filled for baseline modeling.
- Categorical missing values are set to `Unknown` to preserve rows.


In [ ]:
def add_date_parts(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['Year'] = out['Date'].dt.year
    out['Month'] = out['Date'].dt.month
    out['Day'] = out['Date'].dt.day
    out['WeekOfYear'] = out['Date'].dt.isocalendar().week.astype(int)
    out['IsMonthStart'] = out['Date'].dt.is_month_start.astype(int)
    out['IsMonthEnd'] = out['Date'].dt.is_month_end.astype(int)
    return out

def prep_common_features(df: pd.DataFrame) -> pd.DataFrame:
    out = add_date_parts(df)
    for col in ['CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2SinceWeek', 'Promo2SinceYear']:
        if col in out.columns:
            out[col] = out[col].fillna(0)

    for col in ['PromoInterval', 'StateHoliday', 'StoreType', 'Assortment']:
        if col in out.columns:
            out[col] = out[col].fillna('Unknown')

    return out

In [ ]:
# Restrict EDA target analysis to days where stores were open and sales were positive
train_m = prep_common_features(train_m)
test_m = prep_common_features(test_m)

train_open = train_m[(train_m['Open'] == 1) & (train_m['Sales'] > 0)].copy()
print('Rows used for sales EDA (open + positive sales):', train_open.shape[0])

## Core EDA Analyses Performed
1. Target distribution in raw and log scale.
2. Daily sales trend and 30-day smoothing.
3. Day-of-week and promo uplift behavior.
4. Holiday/store-type/assortment effects.
5. Correlation structure across numeric drivers.


In [ ]:
# Sales distribution and log-sales distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(train_open['Sales'], bins=80, kde=True, ax=ax[0])
ax[0].set_title('Sales Distribution')
sns.histplot(np.log1p(train_open['Sales']), bins=80, kde=True, ax=ax[1], color='tab:orange')
ax[1].set_title('Log(1 + Sales) Distribution')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_sales_distribution.png', dpi=140)
plt.show()

In [ ]:
# Aggregate to daily total sales to inspect trend and medium-term seasonality
# Daily total sales trend
daily_sales = train_open.groupby('Date', as_index=False)['Sales'].sum().sort_values('Date')
daily_sales['Rolling30'] = daily_sales['Sales'].rolling(30, min_periods=1).mean()

plt.figure(figsize=(14, 4))
plt.plot(daily_sales['Date'], daily_sales['Sales'], alpha=0.4, label='Daily Total')
plt.plot(daily_sales['Date'], daily_sales['Rolling30'], color='black', linewidth=2, label='30-Day Mean')
plt.title('Daily Sales Trend')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_daily_sales_trend.png', dpi=140)
plt.show()

In [ ]:
# Sales by weekday
plt.figure(figsize=(8, 4))
sns.boxplot(data=train_open, x='DayOfWeek', y='Sales')
plt.title('Sales by Day Of Week')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_sales_by_dayofweek.png', dpi=140)
plt.show()

In [ ]:
# Promo effect
plt.figure(figsize=(7, 4))
sns.boxplot(data=train_open, x='Promo', y='Sales')
plt.title('Promo vs Sales')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_promo_effect.png', dpi=140)
plt.show()

train_open.groupby('Promo')['Sales'].describe()

In [ ]:
# State holiday effect
holiday_order = sorted(train_open['StateHoliday'].astype(str).unique())
plt.figure(figsize=(8, 4))
sns.boxplot(data=train_open, x=train_open['StateHoliday'].astype(str), y='Sales', order=holiday_order)
plt.title('StateHoliday vs Sales')
plt.xlabel('StateHoliday')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_stateholiday_effect.png', dpi=140)
plt.show()

In [ ]:
# Store type / assortment effects
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=train_open, x='StoreType', y='Sales', estimator=np.mean, errorbar=None, ax=ax[0])
ax[0].set_title('Avg Sales by StoreType')
sns.barplot(data=train_open, x='Assortment', y='Sales', estimator=np.mean, errorbar=None, ax=ax[1])
ax[1].set_title('Avg Sales by Assortment')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_storetype_assortment_effects.png', dpi=140)
plt.show()

In [ ]:
# Correlation matrix helps identify first-order linear relationships and leakage risks
# Numeric correlation heatmap
num_cols = ['Sales', 'Customers', 'Promo', 'SchoolHoliday', 'CompetitionDistance', 'DayOfWeek', 'Month', 'WeekOfYear']
corr = train_open[num_cols].corr(numeric_only=True)

plt.figure(figsize=(9, 6))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_correlation_heatmap.png', dpi=140)
plt.show()

In [ ]:
# Store-level ranking snapshot
store_rank = train_open.groupby('Store', as_index=False)['Sales'].mean().sort_values('Sales', ascending=False)
print('Top 10 stores by average sales')
display(store_rank.head(10))
print('Bottom 10 stores by average sales')
display(store_rank.tail(10))

## Output Artifacts
- Figures are exported to `reports/figures/` for reporting reuse.
- Baseline cleaned datasets are exported to `data/processed/` for subsequent notebooks.


In [ ]:
# Persist cleaned datasets so downstream notebooks use a consistent baseline snapshot
# Create baseline cleaned datasets
train_clean = train_m[(train_m['Open'] == 1) & (train_m['Sales'] > 0)].copy()
test_clean = test_m.copy()

train_clean.to_csv(PROCESSED_DIR / 'train_cleaned.csv', index=False)
test_clean.to_csv(PROCESSED_DIR / 'test_cleaned.csv', index=False)

print('Saved:', PROCESSED_DIR / 'train_cleaned.csv', train_clean.shape)
print('Saved:', PROCESSED_DIR / 'test_cleaned.csv', test_clean.shape)

## Next Steps
- Add target transformations (e.g., RMSPE-aligned modeling objective).
- Build lag/rolling features by store for stronger forecasting performance.
- Continue with notebook `02_Statistical_Analysis.ipynb`.